# The Algebra Engine## PDP Skills Demo 2B (15%) / Maths for IT Assignment 1B (18%)In this notebook we are going to build something rather satisfying: a small engine that can work with algebraic expressions. By the end, we will have functions that evaluate, expand, factor, and solve equations -- all written from scratch.Along the way, we will visit the different number systems that mathematics uses, play with sets and their operations, and discover how the algebra we learned in school translates into algorithms.Same ground rules: static resources welcome, thinking in markdown, test everything.

## Part 1: Number Domains and Arithmetic (6 marks)Mathematicians organise numbers into nested families:- **N** (natural numbers): 0, 1, 2, 3, ...- **Z** (integers): ..., -2, -1, 0, 1, 2, ...- **Q** (rationals): any number that can be written as a fraction p/q- **R** (reals): all points on the number line, including irrationals like the square root of 2Each family extends the one before it. Every natural number is an integer, every integer is a rational (just put it over 1), and every rational is a real.

### Your turnLet's write a function `classify_number(n)` that takes a number and returns which domains it belongs to. For instance, `classify_number(3)` might return `["N", "Z", "Q", "R"]`, while `classify_number(-2.5)` returns `["Q", "R"]` and `classify_number(3.14159...)` returns `["R"]`.Think about how to detect whether a number is an integer, whether it is rational (tricky with floating point -- for our purposes, treat anything with a finite decimal representation as rational), and so on.

In [1]:
import math

def classify_number(n):
    """Return a list of the number domains (N, Z, Q, R) that n belongs to."""
    domains = []

    if isinstance(n, int):
        if n >= 0:
            domains.append("N")
        domains.extend(["Z", "Q", "R"])
        return domains

    if isinstance(n, float):
        if n.is_integer():
            if n >= 0:
                domains.append("N")
            domains.extend(["Z", "Q", "R"])
            return domains

        # Floating point cannot represent most irrationals exactly, but we
        # can catch the well-known ones by comparing against a small
        # tolerance. Anything else with a finite decimal representation we
        # treat as rational, as the exercise suggests.
        known_irrationals = [math.pi, math.e, math.sqrt(2), math.sqrt(3), math.sqrt(5)]
        for irrational in known_irrationals:
            if abs(n - irrational) < 1e-4:
                domains.append("R")
                return domains

        domains.extend(["Q", "R"])
        return domains

    return domains


In [2]:
# Test with: 7, -3, 0.5, -2.5, 3.14159265358979...
for n in [7, -3, 0.5, -2.5, 3.14159265358979]:
    print(n, classify_number(n))


7 ['N', 'Z', 'Q', 'R']
-3 ['Z', 'Q', 'R']
0.5 ['Q', 'R']
-2.5 ['Q', 'R']
3.14159265358979 ['R']


### Indices and logarithmsTwo operations that come up constantly in computing:**Indices** (powers): $a^n$ means multiply a by itself n times. Key rules:- $a^m \times a^n = a^{m+n}$- $(a^m)^n = a^{m \times n}$- $a^0 = 1$- $a^{-n} = \frac{1}{a^n}$**Logarithms** are the inverse: if $a^n = x$, then $\log_a(x) = n$.Write a function `power(base, exponent)` that handles positive, negative, and zero exponents without using Python's `**` operator. Then write `log_base(x, base)` that finds the logarithm by repeated division (this will give an integer result for perfect powers, which is fine for our purposes).

In [3]:
def power(base, exponent):
    """Raise base to exponent (positive, negative, or zero) without **."""
    if exponent == 0:
        return 1
    if exponent > 0:
        result = 1
        for _ in range(exponent):
            result *= base
        return result
    result = 1
    for _ in range(-exponent):
        result *= base
    return 1 / result


In [4]:
def log_base(x, base):
    """Find n such that base**n == x, by repeated division."""
    n = 0
    while x > 1:
        x /= base
        n += 1
    return n


In [5]:
# Test both
print(power(2, 5), "expected 32")
print(power(2, 0), "expected 1")
print(power(2, -3), "expected 0.125")
print(log_base(1000, 10), "expected 3")
print(log_base(32, 2), "expected 5")


32 expected 32
1 expected 1
0.125 expected 0.125
3 expected 3
5 expected 5


### Practical geometryFormulas for area, perimeter, volume, and surface area are really just functions: they take measurements as input and return a computed value.Write a small collection of geometry functions. At minimum:- `circle_area(radius)` and `circle_perimeter(radius)`- `rectangle_area(length, width)` and `rectangle_perimeter(length, width)`- `triangle_area(base, height)`- `cylinder_volume(radius, height)` and `sphere_volume(radius)`Use `math.pi` for pi. These are straightforward, but they give us practice writing clean, documented functions with clear parameter names.

In [6]:
import math

def circle_area(radius):
    """Return the area of a circle with the given radius."""
    return math.pi * radius ** 2


def circle_perimeter(radius):
    """Return the circumference of a circle with the given radius."""
    return 2 * math.pi * radius


def rectangle_area(length, width):
    """Return the area of a rectangle."""
    return length * width


def rectangle_perimeter(length, width):
    """Return the perimeter of a rectangle."""
    return 2 * (length + width)


def triangle_area(base, height):
    """Return the area of a triangle given its base and height."""
    return 0.5 * base * height


def cylinder_volume(radius, height):
    """Return the volume of a cylinder."""
    return circle_area(radius) * height


def sphere_volume(radius):
    """Return the volume of a sphere."""
    return (4 / 3) * math.pi * radius ** 3


In [7]:
# Test them with known values
# A circle with radius 5 has area approximately 78.54
print("circle_area(5):", round(circle_area(5), 2))
print("circle_perimeter(5):", round(circle_perimeter(5), 2))
print("rectangle_area(4, 6):", rectangle_area(4, 6))
print("rectangle_perimeter(4, 6):", rectangle_perimeter(4, 6))
print("triangle_area(4, 6):", triangle_area(4, 6))
print("cylinder_volume(2, 5):", round(cylinder_volume(2, 5), 2))
print("sphere_volume(3):", round(sphere_volume(3), 2))


circle_area(5): 78.54
circle_perimeter(5): 31.42
rectangle_area(4, 6): 24
rectangle_perimeter(4, 6): 20
triangle_area(4, 6): 12.0
cylinder_volume(2, 5): 62.83
sphere_volume(3): 113.1


## Part 2: Expressions, Equations, and Polynomials (10 marks)Here is an important distinction: an *expression* is a mathematical phrase that has a value (like `3x + 7`), while an *equation* is a statement that two expressions are equal (like `3x + 7 = 22`). We evaluate expressions; we solve equations.

### Representing polynomialsA polynomial like $3x^2 + 5x - 2$ can be represented as a list of coefficients: `[3, 5, -2]`, where the first element is the coefficient of the highest power. Or we could go the other way: `[-2, 5, 3]`, where position i holds the coefficient of $x^i$. Let's use the second convention -- it is more natural for computation because the index matches the exponent.So `[-2, 5, 3]` means $-2 + 5x + 3x^2$.

### Your turnWrite a function `evaluate_poly(coeffs, x)` that takes a list of coefficients (lowest power first) and a value of x, and returns the value of the polynomial at that point.Then write `poly_to_string(coeffs)` that produces a human-readable string like `"3x^2 + 5x - 2"` from the coefficient list. This is trickier than it sounds -- think about handling zero coefficients, negative coefficients, and the special cases for $x^0$ and $x^1$.

In [8]:
def evaluate_poly(coeffs, x):
    """Evaluate a polynomial (lowest power first) at a given value of x."""
    total = 0
    for power_index, coeff in enumerate(coeffs):
        total += coeff * x ** power_index
    return total


In [9]:
def poly_to_string(coeffs):
    """Produce a human-readable string for a polynomial (lowest power first)."""
    terms = []
    for power_index in range(len(coeffs) - 1, -1, -1):
        coeff = coeffs[power_index]
        if coeff == 0:
            continue
        if power_index == 0:
            term = f"{abs(coeff)}"
        elif power_index == 1:
            term = "x" if abs(coeff) == 1 else f"{abs(coeff)}x"
        else:
            term = f"x^{power_index}" if abs(coeff) == 1 else f"{abs(coeff)}x^{power_index}"

        if not terms:
            terms.append(f"-{term}" if coeff < 0 else term)
        else:
            terms.append(f"- {term}" if coeff < 0 else f"+ {term}")

    return " ".join(terms) if terms else "0"


In [10]:
# Test: evaluate_poly([-2, 5, 3], 2) should give -2 + 10 + 12 = 20
print(evaluate_poly([-2, 5, 3], 2))
print(poly_to_string([-2, 5, 3]))
print(poly_to_string([0, 1, 0, 2]))


20
3x^2 + 5x - 2
2x^3 + x


### Adding and multiplying polynomialsIf polynomials are lists, then adding them is straightforward: add corresponding coefficients. But what about multiplying? Multiplying $(2x + 3)(x + 4)$ requires the "FOIL" method -- or more generally, each term in the first polynomial gets multiplied by each term in the second.Write `add_poly(a, b)` and `multiply_poly(a, b)`.For multiplication, think about what happens to the exponents: when we multiply $a_i x^i$ by $b_j x^j$, we get $a_i b_j x^{i+j}$. So the result coefficient at position $k$ is the sum of all products $a_i \cdot b_j$ where $i + j = k$.

In [11]:
def add_poly(a, b):
    """Add two polynomials given as coefficient lists (lowest power first)."""
    length = max(len(a), len(b))
    result = []
    for i in range(length):
        coeff_a = a[i] if i < len(a) else 0
        coeff_b = b[i] if i < len(b) else 0
        result.append(coeff_a + coeff_b)
    return result


In [12]:
def multiply_poly(a, b):
    """Multiply two polynomials given as coefficient lists (lowest power first)."""
    result = [0] * (len(a) + len(b) - 1)
    for i, coeff_a in enumerate(a):
        for j, coeff_b in enumerate(b):
            result[i + j] += coeff_a * coeff_b
    return result


In [13]:
# Test: (2x + 3)(x + 4) = 2x^2 + 11x + 12
# In our notation: multiply_poly([3, 2], [4, 1]) should give [12, 11, 2]
print(add_poly([-2, 5, 3], [1, 1]))
print(multiply_poly([3, 2], [4, 1]))


[-1, 6, 3]
[12, 11, 2]


### Solving equationsLet's start with the most fundamental: solving a linear equation $ax + b = 0$. The solution is simply $x = -b/a$ (as long as $a \neq 0$).Write `solve_linear(coeffs)` where coeffs is `[b, a]` in our convention.Then the quadratic formula. For $ax^2 + bx + c = 0$:$$x = \frac{-b \pm \sqrt{b^2 - 4ac}}{2a}$$Write `solve_quadratic(coeffs)` where coeffs is `[c, b, a]`. Think about three cases:- The discriminant ($b^2 - 4ac$) is positive: two real roots- The discriminant is zero: one repeated root- The discriminant is negative: no real roots (for now -- complex roots exist but we will keep things real)

In [14]:
def solve_linear(coeffs):
    """Solve ax + b = 0 given coeffs = [b, a]. Returns x."""
    b, a = coeffs
    if a == 0:
        raise ValueError("Not a linear equation: the coefficient of x cannot be 0")
    return -b / a


In [15]:
def solve_quadratic(coeffs):
    """Solve ax^2 + bx + c = 0 given coeffs = [c, b, a]. Returns a tuple of roots."""
    c, b, a = coeffs
    if a == 0:
        return (solve_linear([c, b]),)

    discriminant = b ** 2 - 4 * a * c
    if discriminant > 0:
        root1 = (-b + discriminant ** 0.5) / (2 * a)
        root2 = (-b - discriminant ** 0.5) / (2 * a)
        return (root1, root2)
    elif discriminant == 0:
        return (-b / (2 * a),)
    else:
        return ()


In [16]:
# Test: x^2 - 4x + 3 = 0 should give roots 1 and 3
# In our notation: solve_quadratic([3, -4, 1])
print(solve_quadratic([3, -4, 1]))
print(solve_linear([-6, 2]), "expected 3")


(3.0, 1.0)
3.0 expected 3


### FactorisationIf we can find the roots of a quadratic $ax^2 + bx + c$, we can factor it as $a(x - r_1)(x - r_2)$ where $r_1$ and $r_2$ are the roots. Write a function `factor_quadratic(coeffs)` that returns the factored form as a string, or indicates if the polynomial cannot be factored over the reals.

In [17]:
def factor_quadratic(coeffs):
    """Return the factored form of ax^2 + bx + c as a string, if it has real roots."""
    c, b, a = coeffs
    roots = solve_quadratic(coeffs)
    if len(roots) == 2:
        r1, r2 = roots
        return f"{a}(x - {r1})(x - {r2})"
    elif len(roots) == 1:
        r = roots[0]
        return f"{a}(x - {r})^2"
    return "Cannot be factored over the reals (no real roots)"


In [18]:
# Test with a few quadratics
print(factor_quadratic([3, -4, 1]))
print(factor_quadratic([1, -2, 1]))
print(factor_quadratic([5, 0, 1]))


1(x - 3.0)(x - 1.0)
1(x - 1.0)^2
Cannot be factored over the reals (no real roots)


## Part 3: Sets as Collections (6 marks)A *set* is a collection of distinct elements where order does not matter. The set {1, 2, 3} is the same as {3, 1, 2}. Sets give us a language for talking about collections, membership, and relationships.Python has a built-in `set` type, but let's build the operations ourselves using sorted lists. This connects nicely to the search and sort algorithms from our first notebook.

### Your turnRepresent sets as sorted lists with no duplicates. Write:1. `make_set(items)` -- takes a list, removes duplicates, sorts it, and returns the result2. `is_member(s, item)` -- checks if item is in set s (use binary search if you like!)3. `union(a, b)` -- returns all elements that are in a or b or both4. `intersection(a, b)` -- returns all elements that are in both a and b5. `difference(a, b)` -- returns elements that are in a but not in b6. `symmetric_difference(a, b)` -- returns elements that are in a or b but not bothSince both input lists are sorted, you can implement union and intersection efficiently by walking through both lists simultaneously (like a merge step). Think about how this works before coding it.

In [19]:
def make_set(items):
    """Return a sorted list of items with duplicates removed."""
    result = []
    for item in sorted(items):
        if not result or result[-1] != item:
            result.append(item)
    return result


def is_member(s, item):
    """Check whether item is present in the sorted list s, using binary search."""
    low, high = 0, len(s) - 1
    while low <= high:
        mid = (low + high) // 2
        if s[mid] == item:
            return True
        elif s[mid] < item:
            low = mid + 1
        else:
            high = mid - 1
    return False


def union(a, b):
    """Return all elements in a or b (or both), merging two sorted lists."""
    result = []
    i = j = 0
    while i < len(a) and j < len(b):
        if a[i] < b[j]:
            result.append(a[i])
            i += 1
        elif a[i] > b[j]:
            result.append(b[j])
            j += 1
        else:
            result.append(a[i])
            i += 1
            j += 1
    result.extend(a[i:])
    result.extend(b[j:])
    return result


def intersection(a, b):
    """Return elements that are in both a and b, merging two sorted lists."""
    result = []
    i = j = 0
    while i < len(a) and j < len(b):
        if a[i] < b[j]:
            i += 1
        elif a[i] > b[j]:
            j += 1
        else:
            result.append(a[i])
            i += 1
            j += 1
    return result


def difference(a, b):
    """Return elements that are in a but not in b."""
    return [item for item in a if not is_member(b, item)]


def symmetric_difference(a, b):
    """Return elements that are in a or b, but not both."""
    return sorted(difference(a, b) + difference(b, a))


In [20]:
# Test with some examples
a = make_set([3, 1, 4, 1, 5, 9, 2, 6])
b = make_set([5, 7, 2, 8, 1, 8])
print("a:", a)
print("b:", b)
print("union:", union(a, b))
print("intersection:", intersection(a, b))
print("difference a-b:", difference(a, b))
print("difference b-a:", difference(b, a))
print("symmetric_difference:", symmetric_difference(a, b))
print("is_member(a, 4):", is_member(a, 4))
print("is_member(a, 100):", is_member(a, 100))


a: [1, 2, 3, 4, 5, 6, 9]
b: [1, 2, 5, 7, 8]
union: [1, 2, 3, 4, 5, 6, 7, 8, 9]
intersection: [1, 2, 5]
difference a-b: [3, 4, 6, 9]
difference b-a: [7, 8]
symmetric_difference: [3, 4, 6, 7, 8, 9]
is_member(a, 4): True
is_member(a, 100): False


### Sets in actionSets are not just abstract. Think about how a search engine might use intersection: if you search for "python sorting algorithms," the engine finds the set of pages containing "python," the set containing "sorting," and the set containing "algorithms," then returns their intersection.Or think about a music app recommending songs: it might take the union of genres you listen to and the intersection of that with trending tracks.Can you think of another practical application? Write a brief example below.

### Your example

### Solving inequalities and simultaneous equationsLet's return to equations briefly. A linear inequality like $2x + 3 > 7$ defines a set of solutions: all x values satisfying the condition. Can you write a function `solve_linear_inequality(a, b, c)` that solves $ax + b > c$ and returns a description of the solution set?Then, simultaneous equations: given two linear equations with two unknowns, find the values of x and y that satisfy both. The classic approach uses substitution or elimination.Write `solve_simultaneous(eq1, eq2)` where each equation is represented as `[a, b, c]` meaning $ax + by = c$.

In [21]:
def solve_linear_inequality(a, b, c):
    """Solve ax + b > c, returning a description of the solution set."""
    if a == 0:
        return "All real numbers" if b > c else "No solution"
    boundary = (c - b) / a
    if a > 0:
        return f"x > {boundary}"
    return f"x < {boundary}"


In [22]:
def solve_simultaneous(eq1, eq2):
    """
    Solve two simultaneous linear equations ax + by = c, using elimination.
    Each equation is given as [a, b, c].
    """
    a1, b1, c1 = eq1
    a2, b2, c2 = eq2
    determinant = a1 * b2 - a2 * b1
    if determinant == 0:
        return None  # No unique solution: the lines are parallel or identical
    x = (c1 * b2 - c2 * b1) / determinant
    y = (a1 * c2 - a2 * c1) / determinant
    return (x, y)


In [23]:
# Test: solve the system x + y = 10, 2x - y = 5
# Should give x = 5, y = 5
print(solve_simultaneous([1, 1, 10], [2, -1, 5]))
print(solve_linear_inequality(2, 3, 7), "expected x > 2.0")


(5.0, 5.0)
x > 2.0 expected x > 2.0


## Wrapping UpWe have built a genuine algebra engine: polynomial representation, evaluation, arithmetic, root-finding, factorisation, set operations, and equation solving. Each piece is a function that can be tested independently and composed with others.Reflect on what you have built:- Which mathematical concept became clearer through programming it?- Where did the translation from formula to code surprise you?- What connections do you see between the set operations and the programming concepts we have been learning?

### Your reflection

---*This notebook serves as both PDP Skills Demo 2B (15%) and Maths for IT Assignment 1B (18%). Programming skills assessed include modularisation, functions with parameters and return values, documented code, testing, and coding standards. Mathematical skills assessed include number domains, indices and logarithms, area/perimeter/volume formulas, algebraic expressions, polynomial operations, quadratic equations, factorisation, linear inequalities, simultaneous equations, and set operations.*